# Week 11 – Vector Embeddings with ChromaDB

## Objective
To generate vector embeddings using Sentence Transformers, store them in ChromaDB, and perform semantic similarity searches.

In [1]:
import os
import chromadb
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
import chromadb
import sentence_transformers
import pandas
import numpy

print("All libraries are ready!")

All libraries are ready!


In [3]:

import os

document_folder = "../dataset/documents"

documents = []
document_ids = []

for filename in os.listdir(document_folder):
    if filename.endswith(".txt"):
        with open(os.path.join(document_folder, filename), "r", encoding="utf-8") as file:
            documents.append(file.read())
            document_ids.append(filename)

print("Documents loaded successfully!")
print("Total documents:", len(documents))


Documents loaded successfully!
Total documents: 10


In [4]:

# Load Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
embeddings = model.encode(documents)

# Display results
print("Embeddings generated successfully!")
print("Number of documents:", len(embeddings))
print("Embedding dimensions:", embeddings.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings generated successfully!
Number of documents: 10
Embedding dimensions: (10, 384)


In [5]:
# Create persistent ChromaDB client
client = chromadb.PersistentClient(path="../chroma_db")

# Create collection
collection = client.get_or_create_collection(
    name="ai_documents"
)

print("ChromaDB collection created successfully!")

ChromaDB collection created successfully!


In [6]:

# Add metadata
metadatas = [
    {
        "title": doc_id,
        "category": "Artificial Intelligence",
        "source": doc_id
    }
    for doc_id in document_ids
]

# Store documents and embeddings
collection.upsert(
    ids=document_ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Documents stored successfully!")
print("Total stored documents:", collection.count())


Documents stored successfully!
Total stored documents: 10


In [7]:

# Sample user queries
queries = [
    "What is Artificial Intelligence?",
    "Explain Machine Learning.",
    "What is Deep Learning?",
    "How do computers understand human language?",
    "What are Neural Networks?",
    "Explain Computer Vision.",
    "How are robots used in industries?",
    "What are Expert Systems?",
    "What is Generative AI?",
    "How is AI used in healthcare?"
]

# Perform similarity search
for query in queries:
    print("\nQuery:", query)

    query_embedding = model.encode([query])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=5
    )

    for i, doc in enumerate(results["documents"][0]):
        print(f"\nRank {i+1}:")
        print(doc[:200])
        print("Distance:", results["distances"][0][i])



Query: What is Artificial Intelligence?

Rank 1:
Artificial Intelligence (AI)

Artificial Intelligence is a branch of computer science that enables machines to perform tasks that normally require human intelligence. AI systems can learn from data, s
Distance: 0.3629903793334961

Rank 2:
Robotics and Artificial Intelligence

Robotics combines engineering and Artificial Intelligence to create intelligent machines. AI-powered robots can perform tasks automatically, recognize objects, an
Distance: 0.7340501546859741

Rank 3:
Applications of Artificial Intelligence

Artificial Intelligence is used in many real-world applications. In healthcare, AI helps doctors detect diseases. In education, it supports personalized learni
Distance: 0.8639258742332458

Rank 4:
Generative Artificial Intelligence

Generative AI is a type of Artificial Intelligence that can create new content such as text, images, music, and videos. It uses advanced machine learning models to 
Distance: 1.0113346576690674

Ra

In [8]:

import pandas as pd

retrieved_data = []

for query in queries:

    query_embedding = model.encode([query])

    search_results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=5
    )

    for i in range(5):
        retrieved_data.append({
            "query": query,
            "rank": i + 1,
            "document_id": search_results["ids"][0][i],
            "document": search_results["documents"][0][i],
            "distance": search_results["distances"][0][i]
        })

# Create DataFrame
df = pd.DataFrame(retrieved_data)

# Save CSV file
df.to_csv("../dataset/retrieved_results.csv", index=False)

print("Retrieval results saved successfully!")
print("Total results:", len(df))

df.head(10)


Retrieval results saved successfully!
Total results: 50


,query,rank,document_id,document,distance
0,What is Artificial Intelligence?,1,document1.txt,Artificial Intelligence (AI)\n\nArtificial Int...,0.362990
1,What is Artificial Intelligence?,2,document7.txt,Robotics and Artificial Intelligence\n\nRoboti...,0.734050
2,What is Artificial Intelligence?,3,document10.txt,Applications of Artificial Intelligence\n\nArt...,0.863926
3,What is Artificial Intelligence?,4,document9.txt,Generative Artificial Intelligence\n\nGenerati...,1.011335
4,What is Artificial Intelligence?,5,document2.txt,Machine Learning\n\nMachine Learning is a subs...,1.038431
5,Explain Machine Learning.,1,document2.txt,Machine Learning\n\nMachine Learning is a subs...,0.695337
6,Explain Machine Learning.,2,document6.txt,Computer Vision\n\nComputer Vision is a field ...,1.135684
7,Explain Machine Learning.,3,document1.txt,Artificial Intelligence (AI)\n\nArtificial Int...,1.143528
8,Explain Machine Learning.,4,document3.txt,Deep Learning\n\nDeep Learning is a part of Ma...,1.147046
9,Explain Machine Learning.,5,document10.txt,Applications of Artificial Intelligence\n\nArt...,1.171181


In [9]:

# Keyword-based search function
def keyword_search(query, documents, document_ids):
    scores = []

    for doc_id, doc in zip(document_ids, documents):
        score = sum(
            word.lower() in doc.lower()
            for word in query.split()
        )

        scores.append((doc_id, score))

    scores.sort(key=lambda x: x[1], reverse=True)

    return scores[:5]


# Compare both search methods
for query in queries[:3]:

    print("\nQuery:", query)

    print("\nSemantic Search:")
    semantic_results = collection.query(
        query_embeddings=model.encode([query]).tolist(),
        n_results=5
    )

    for doc_id in semantic_results["ids"][0]:
        print(doc_id)

    print("\nKeyword Search:")
    keyword_results = keyword_search(
        query, documents, document_ids
    )

    for doc_id, score in keyword_results:
        print(doc_id, "Score:", score)



Query: What is Artificial Intelligence?

Semantic Search:
document1.txt
document7.txt
document10.txt
document9.txt
document2.txt

Keyword Search:
document1.txt Score: 2
document10.txt Score: 2
document2.txt Score: 2
document3.txt Score: 2
document4.txt Score: 2

Query: Explain Machine Learning.

Semantic Search:
document2.txt
document6.txt
document1.txt
document3.txt
document10.txt

Keyword Search:
document2.txt Score: 2
document1.txt Score: 1
document10.txt Score: 1
document3.txt Score: 1
document5.txt Score: 1

Query: What is Deep Learning?

Semantic Search:
document3.txt
document5.txt
document2.txt
document1.txt
document6.txt

Keyword Search:
document3.txt Score: 2
document5.txt Score: 2
document1.txt Score: 1
document10.txt Score: 1
document2.txt Score: 1


## Observations

1. Semantic search retrieves documents based on meaning and context.
2. Keyword search retrieves documents based on exact word matching.
3. Semantic search can find relevant documents even when different words are used.
4. Keyword search may miss relevant documents if the exact words are not present.
5. ChromaDB provides efficient similarity-based document retrieval.

## Conclusion

Vector embeddings were successfully generated using Sentence Transformers and stored in ChromaDB. Similarity search was performed to retrieve the Top-5 relevant documents for each query. This knowledge base can be used for Retrieval-Augmented Generation (RAG).